# Diseño de aplicaciones de IA con LLM
## Módulo: Seguridad y cumplimiento

- **Institución:** BSG Institute
- **Curso:** Diseño de aplicaciones de IA con LLM
- **Módulo:** Seguridad y cumplimiento de datos
- **Docente:** Jorge I. Blanco

En este notebook vamos a usar **Named Entity Recognition (NER)** en español para detectar datos personales y sensibles en texto, y conectar estos resultados con marcos regulatorios como:

- **GDPR** (*General Data Protection Regulation*), aplicable a datos personales de personas en la Unión Europea.
- **Ley 1581 de 2012** en Colombia, sobre protección de datos personales.
- **LFPDPPP** en México (*Ley Federal de Protección de Datos Personales en Posesión de los Particulares*).
- **Ley 29733** en Perú, Ley de Protección de Datos Personales.

La idea es avanzar desde ejemplos simples (detectar nombres, organizaciones y lugares) hasta textos de dominios sensibles como salud y legal, manteniendo el enfoque en **seguridad y cumplimiento**.

## Índice

- [1. Marco de privacidad: GDPR y leyes locales](#sec-marco-privacidad)
- [2. Recordatorio técnico: Token classification y NER](#sec-recordatorio-tecnico)
- [3. Selección de modelo NER en español](#sec-modelo-ner)
- [4. Ejercicio 1 – NER básico en español](#sec-ejercicio-1)
- [5. Ejercicio 2 – Anonimización básica con NER](#sec-ejercicio-2)
- [6. Ejercicio 3 – Frases con mayor complejidad](#sec-ejercicio-3)
- [7. Ejercicio 4 – Rellenar máscaras con `fill-mask`](#sec-ejercicio-4)
- [8. Ejercicio 5 – NER aplicado a salud / medicina](#sec-ejercicio-5)
- [9. Ejercicio 6 – NER aplicado a documentos legales](#sec-ejercicio-6)
- [10. Ejercicio 7 – Selección autónoma de modelo y mini-experimento](#sec-ejercicio-7)
- [11. Ejercicio 8 – Caso de estudio de cumplimiento](#sec-ejercicio-8)
- [12. Recursos y datasets recomendados](#sec-recursos)
- [13. Cierre y reflexión final](#sec-cierre)


<a id="sec-marco-privacidad"></a>
## 2. Marco de privacidad: GDPR y leyes locales

Antes de escribir una sola línea de código, es importante tener claro **por qué** estamos haciendo todo esto.

### 2.1. GDPR (General Data Protection Regulation)

El **GDPR** es el reglamento europeo de protección de datos personales. Busca dos cosas principales:
- Dar a las personas **más control** sobre cómo se usan sus datos personales.
- Obligar a las organizaciones a aplicar principios claros de tratamiento (minimización, limitación de finalidad, seguridad, transparencia, etc.).

Algunas ideas clave:

- **Dato personal**: cualquier información que identifique o pueda identificar a una persona (nombre, correo, documento, IP, etc.).
- **Dato sensible**: categorías especiales como salud, orientación sexual, opiniones políticas, religión, etc.
- **Minimización de datos**: solo recolectar y procesar lo estrictamente necesario para la finalidad declarada.
- **Seguridad**: aplicar medidas técnicas y organizativas adecuadas (cifrado, control de acceso, anonimización, pseudonimización, auditoría, etc.).

> En este notebook, usaremos NER como **una herramienta técnica** que puede ayudarnos a localizar y anonimizar datos personales en textos antes de enviarlos a modelos de lenguaje o pipelines más complejos.

### 2.2. Colombia – Ley 1581 de 2012

En Colombia, la **Ley 1581 de 2012** establece disposiciones generales para la protección de datos personales.

Puntos básicos para este módulo:

- Reconoce el derecho de todas las personas a conocer, actualizar y rectificar sus datos.
- Define principios como **legalidad, finalidad, libertad, veracidad, transparencia, acceso y circulación restringida, seguridad y confidencialidad**.
- Exige a las organizaciones adoptar medidas de seguridad y políticas internas de tratamiento de datos.

### 2.3. México – LFPDPPP

En México, la referencia clave es la **Ley Federal de Protección de Datos Personales en Posesión de los Particulares (LFPDPPP)**.

Elementos a tener presentes:

- Aplica al sector privado que trata datos personales.
- Se basa en principios como **consentimiento**, **finalidad**, **proporcionalidad**, **calidad de los datos** y **responsabilidad**.

### 2.4. Perú – Ley 29733

En Perú, la **Ley 29733** regula el tratamiento de datos personales en bancos de datos, tanto públicos como privados.


Aspectos clave:

- Establece principios como **legalidad, consentimiento, proporcionalidad, calidad, seguridad y disposición de recurso**.

- Crea la Autoridad Nacional de Protección de Datos Personales como ente supervisor.


### 2.5. Conexión con NER y LLM

En todos estos marcos regulatorios hay ideas comunes:

- Proteger datos personales.
- Limitar el uso de esos datos a finalidades legítimas.
- Implementar medidas de seguridad para reducir riesgos.

En las siguientes secciones usaremos NER en español como **mecanismo de apoyo** para:

- Detectar datos personales en texto.
- Anonimizar o pseudonimizar antes de enviar información a un LLM.
- Discutir dónde NER ayuda y dónde no es suficiente.

<a id="sec-recordatorio-tecnico"></a>
## 3. Recordatorio técnico: Token classification y NER

La tarea de **token classification** asigna una etiqueta a algunos tokens dentro de un texto.  
NER (Named Entity Recognition) es un caso particular donde esas etiquetas indican entidades como:

- Personas (PER)
- Organizaciones (ORG)
- Lugares (LOC)
- Otros tipos según el modelo (MISC, enfermedades, medicamentos, artículos legales, etc.)

En Hugging Face, usamos normalmente:

- El método `pipeline("ner")` para inferencia rápida.
- El parámetro `aggregation_strategy="simple"` para agrupar tokens contiguos en entidades completas (por ejemplo, unir “Jorge” + “Ignacio”+ "Blanco").

Diagrama conceptual:

1. Texto de entrada (string en español).  
2. Tokenización y paso por el modelo NER.  
3. Salida por token (B-PER, I-PER, B-ORG, etc.).  
4. Agregación en entidades:  
   - `entity_group`: PER, ORG, LOC, etc.  
   - `word`: texto de la entidad.  
   - `start`, `end`: posiciones de la entidad en el texto.  
   - `score`: confianza del modelo.

En las secciones siguientes haremos:

- Carga de un modelo NER general en español.
- Experimentos con textos que contienen datos personales.
- Anonimización básica usando las entidades detectadas.

<a id="sec-modelo-ner"></a>
## 4. Selección de modelo NER en español

Trabajaremos con modelos de Hugging Face ya entrenados para NER en español.

### 4.1. Catálogo de modelos sugerido

Busquen y revisen lo modelos NER en español en:

- [https://huggingface.co/models?pipeline_tag=token-classification&language=es&search=ner](https://huggingface.co/models?pipeline_tag=token-classification&language=es&search=ner)

Verás modelos:

- Generales (textos noticiosos, CoNLL-2002 en español).
- Multilingües (por ejemplo XLM-R).
- Específicos de dominios como salud o legal.

### 4.2. Modelo base recomendado para empezar

En este notebook utilizaremos como modelo base:

- `MMG/xlm-roberta-large-ner-spanish` – modelo NER en español entrenado sobre la porción española de CoNLL-2002.

En una celda de código, pueden cargarlo así:

```python
from transformers import pipeline

modelo_base = "MMG/xlm-roberta-large-ner-spanish"

ner_es = pipeline(
    "ner",
    model=modelo_base,
    aggregation_strategy="simple"
)

texto_demo = "Mi nombre es Jorge Blanco, vivo en Bogotá y trabajo para BSG Institute."
# luego en una lista carguen el nombre de los integrantes del gurpo, donde viven y en donde trabajan
ner_es(texto_demo)
```

A partir de aquí, `ner_es` será nuestro pipeline principal para los primeros ejercicios.

<a id="sec-ejercicio-1"></a>
## 5. Ejercicio 1 – NER básico en español


**Objetivo:**  
Comprobar que el modelo NER en español detecta correctamente personas, organizaciones y lugares en frases simples, y relacionar cada entidad con la idea de “dato personal”.

### 5.1. Código base

Usa algo como lo siguiente en una celda de código:

```python
texto_1 = "aqui coloquen el nombre de los integrantes del grupo + la otra informacion pedida en el ejercicio anterior"
resultado_1 = ner_es(texto_1)
resultado_1
```

### 5.2. Preguntas a resolver

1. ¿Qué entidades aparecen en la salida (campo `entity_group`)?  
2. ¿Qué texto exacto detectó el modelo para cada entidad (`word`)?  
3. ¿Cuáles de esas entidades son claramente **datos personales** (PII)?  
4. ¿Qué implicaciones tendría enviar este texto a un LLM externo sin ningún tipo de anonimización?

> Nota: este ejercicio sirve para fijar la idea de que NER puede ser la “primera capa” de detección de datos personales en un pipeline de IA.

<a id="sec-ejercicio-2"></a>
## 6. Ejercicio 2 – Anonimización básica con NER

**Objetivo:**  
Usar la salida de NER para generar una versión anonimizada del texto, sustituyendo las entidades por etiquetas genéricas. Esto se conecta directamente con los principios de minimización y protección de datos de GDPR y de las leyes locales.

### 6.1. Función de anonimización

En una celda de código, define una función como esta:

```python
def anonimizar_entidades(texto, entidades):
    # Ordenar las entidades de atrás hacia adelante para no romper índices
    entidades_ordenadas = sorted(entidades, key=lambda x: x["start"], reverse=True)
    for ent in entidades_ordenadas:
        etiqueta = ent["entity_group"]
        inicio = ent["start"]
        fin = ent["end"]
        texto = texto[:inicio] + f"[{etiqueta}]" + texto[fin:]
    return texto

texto_anon = anonimizar_entidades(texto_1, resultado_1)

print("Texto original:")
print(texto_1)
print("\nTexto anonimizado:")
print(texto_anon)
```

### 6.2. Preguntas de reflexión

- ¿El texto anonimizado sigue siendo útil para tareas de análisis semántico?  
- ¿Crees que un LLM podría entender igual el contexto con `[PER]`, `[ORG]` y `[LOC]`?  
- ¿Qué pasa si el modelo NER no detecta alguna entidad importante (por ejemplo, un número de documento o un correo)?

> En un flujo de cumplimiento real, la anonimización basada en NER se complementa con reglas adicionales y revisión humana.

<a id="sec-ejercicio-3"></a>
## 7. Ejercicio 3 – Frases con mayor complejidad

**Objetivo:**  
Ver cómo se comporta el modelo NER en frases más realistas, con combinaciones de personas, organizaciones, lugares y conceptos propios de salud, legal o finanzas.

### 7.1. Conjunto de frases de prueba



```python
textos = [
    "Carlos trabaja en Microsoft y vive en Madrid.",
    "La paciente Ana Torres fue atendida en la Clínica San Rafael de Lima.",
    "El abogado Juan Pérez firmó el contrato en Ciudad de México para Inversiones Andinas S.A.",
    "El número de historia clínica de María Gómez fue registrado en el hospital central.",
]

for i, t in enumerate(textos, 1):
    print(f"\nTexto {i}")
    print(t)
    print(ner_es(t))
```

### 7.2. Preguntas para el estudiante

- ¿En cuáles textos el modelo funciona bien?  
- ¿En qué casos empieza a fallar (entidades que no detecta o que clasifica mal)?  
- ¿Qué tipos de datos importantes para cumplimiento no aparecen como entidades (ej. “número de historia clínica”)?  
- ¿Qué riesgos ocurren si confiamos solo en NER para anonimización?

<a id="sec-ejercicio-4"></a>
## Ejercicio 4 – Rellenar máscaras con `fill-mask` (Masked Language Modeling)

Además de NER, otra tarea útil para hablar de **sesgos** y **riesgos** en LLM es el relleno de máscaras (*fill-mask*). En esta tarea, el modelo recibe una frase con una palabra oculta y debe predecir qué palabra encaja mejor en ese contexto.

En Hugging Face, esta tarea se usa con el pipeline:

- `pipeline("fill-mask")` para inglés (por defecto o indicando modelo).
- Para español, podemos usar un modelo de lenguaje entrenado con objetivo de *masked language modeling* adecuado al idioma.

> Importante: el token de máscara debe coincidir con el del modelo (por ejemplo, BERT usa `[MASK]`, RoBERTa usa `<mask>`).

### 4.1. Ejemplo básico en español

En una celda de código:

```python
from transformers import pipeline

# Modelo fill-mask multilingüe (ejemplo, puedes cambiarlo por otro apropiado)
fill_mask = pipeline(
    "fill-mask",
    model="dccuchile/bert-base-spanish-wwm-cased"  # modelo BERT en español
)

texto_mask = "Mi nombre es [MASK] y trabajo en una empresa de tecnología."
fill_mask(texto_mask)
```

### 4.2. Observando sesgos en predicciones

Prueben ahora con frases que puedan revelar estereotipos:

```python
oraciones = [
    "El doctor [MASK] atendió a la paciente.",
    "La enfermera [MASK] cuidó al paciente.",
    "El ingeniero [MASK] diseñó el puente.",
    "La secretaria [MASK] organizó la reunión.",
]

for t in oraciones:
    print(f"\nOración: {t}")
    for pred in fill_mask(t):
        print(f"  {pred['sequence']}  (score={pred['score']:.3f})")
    # Limitar a las 5 mejores predicciones por defecto
```

### 4.3. Preguntas de reflexión

- ¿Qué nombres o palabras suele proponer el modelo para cada profesión?
- ¿Detectan estereotipos de género, cultura o rol profesional en las predicciones?  
- ¿Qué implicaciones tiene esto para aplicaciones reales (por ejemplo, asistentes de carrera, generación de ejemplos en educación, etc.)?  
- ¿Cómo conectarían esta observación con las obligaciones de no discriminación y trato justo en marcos como GDPR o las leyes locales?

> Este ejercicio complementa NER: aquí no buscamos identificar entidades, sino entender **cómo el modelo “imagina” el mundo** cuando tiene que completar una frase. Eso nos da material para discutir sesgos y riesgos éticos.

<a id="sec-ejercicio-5"></a>
## 8. Ejercicio 5 – NER aplicado a salud / medicina

El dominio de salud es especialmente muy sensible:

- GDPR lo trata como **categoría especial de dato** (salud).
- Las leyes locales suelen exigir medidas reforzadas para historias clínicas, diagnósticos, medicamentos, etc.

### 8.1. Texto de ejemplo clínico

```python
texto_salud = "La paciente Laura Medina recibió tratamiento con paracetamol en el Hospital Central de Bogotá."
resultado_salud = ner_es(texto_salud)
resultado_salud
```

Probablemente el modelo general detectará personas, hospitales, ciudades… pero **no** etiquetas específicas como “fármaco” o “diagnóstico”.

### 8.2. Recursos de modelos y datasets clínicos

Para ir más allá del modelo general, sugiere explorar:

- Repositorio de modelos biomédicos y clínicos en español:  
  - GitHub: https://github.com/PlanTL-GOB-ES/lm-biomedical-clinical-es  
- Datasets en Hugging Face:  
  - **CANTEMIST NER**: [https://huggingface.co/datasets/PlanTL-GOB-ES/cantemist-ner](https://huggingface.co/datasets/PlanTL-GOB-ES/cantemist-ner)  
  - **PharmaCoNER**: [https://huggingface.co/datasets/PlanTL-GOB-ES/pharmaconer](https://huggingface.co/datasets/PlanTL-GOB-ES/pharmaconer)

### 8.3. Actividad

1. Ejecuten el ejemplo clínico con el modelo general.  
2. Investigen un modelo NER biomédico en español en Hugging Face.  
3. Comparen los resultados de NER general vs. NER biomédico.  
4. Identifiquen qué entidades clínicas importantes aparecen solo con el modelo especializado.

<a id="sec-ejercicio-6"></a>
## 9. Ejercicio 6 – NER aplicado a documentos legales

En el dominio legal, los textos contienen:

- Nombres de personas y empresas.
- Juzgados, ciudades, fechas, números de contrato.
- Referencias a artículos de leyes, resoluciones, sentencias.

Todo esto es relevante para **cumplimiento** y para sistemas que revisan contratos, demandas o políticas.

### 9.1. Texto de ejemplo legal

```python
texto_legal = (
    "El abogado Carlos Ramírez presentó la demanda ante el Juzgado Primero de Lima "
    "el 12 de mayo de 2026 en representación de Grupo Andes S.A.S"
)

resultado_legal = ner_es(texto_legal)
resultado_legal
```

### 9.2. Recursos legales en Hugging Face

- Modelo NER legal en español (ejemplo):  
  - `agomez302/nlp-dr-ner`  
  - Página: [https://huggingface.co/agomez302/nlp-dr-ner](https://huggingface.co/agomez302/nlp-dr-ner)
- Guía de fine-tuning de modelos de clasificación de tokens para datos legales:  
  - [https://huggingface.co/blog/bikashpatra/legal-data-token-classification-fine-tuning](https://huggingface.co/blog/bikashpatra/legal-data-token-classification-fine-tuning)

### 9.3. Actividad

- A partir del resultado de NER general:
  - ¿Qué entidades son importantes para un flujo de revisión legal?  
  - ¿Qué entidades legales relevantes no aparecen (ej. tipo de proceso, número de expediente, referencia a norma)?  
- Discutan y registren entre ustedes qué necesitaría un sistema NER “serio” para legal:
  - Etiquetas más específicas.  
  - Entrenamiento con datasets legales.  
  - Reglas de negocio encima del modelo.

<a id="sec-ejercicio-7"></a>
## 10. Ejercicio 7 – Selección autónoma de modelo y mini-experimento

En este punto, cada grupo debe:

1. Elegir un modelo NER diferente en español desde el catálogo:  
   - [https://huggingface.co/models?pipeline_tag=token-classification&language=es&search=ner](https://huggingface.co/models?pipeline_tag=token-classification&language=es&search=ner)
2. Justificar su elección según:
   - Idioma y dominio (general, biomédico, legal, etc.).  
   - Tamaño del modelo y viabilidad práctica.  
   - Documentación disponible.
   - fecha de publicación del modelo o actulización registrada

### 10.1. Plantilla de código

sugerencia de código:

```python
from transformers import pipeline

# Reemplaza esta línea con el modelo que hayan elijido
modelo_estudiante = "MMG/xlm-roberta-large-ner-spanish"

ner_est = pipeline(
    "ner",
    model=modelo_estudiante,
    aggregation_strategy="simple"
)

texto_prueba = (
    "Juan Pérez autorizó el tratamiento en la Clínica del Norte de Medellín "
    "y firmó el consentimiento informado."
)

ner_est(texto_prueba)
```

### 10.2. Preguntas de análisis

- ¿Qué etiquetas (`entity_group`) devuelve el modelo?  
- ¿Coinciden con las que esperaban por el dominio (general, clínico, legal, etc.)?  
- ¿Cómo cambiarían la estrategia de anonimización dependiendo del tipo de entidades detectadas?

<a id="sec-ejercicio-8"></a>
## 11. Ejercicio 8 – Caso de estudio de cumplimiento

**Objetivo:**  
Diseñar un mini caso realista que conecte NER con cumplimiento normativo en un dominio elegido por cada grupo.

### 11.1. Dominios sugeridos

- **Salud**: historias clínicas, recetas, informes de laboratorio.
- **Legal**: contratos, demandas, resoluciones, expedientes.
- **Educación**: listados de estudiantes, notas, informes de desempeño.
- **Finanzas**: formularios, extractos, reclamaciones, reportes de riesgo.

### 11.2. Entregable sugerido

Para cada grupo, el caso debe incluir:

1. **Texto de ejemplo** del dominio (inventado o anonimizado para clase).  
2. **Modelo NER utilizado** y breve justificación.  
3. **Salida NER** (lista de entidades).  
4. **Texto anonimizado** usando una función tipo `anonimizar_entidades`.  
5. **Riesgos identificados**: qué datos personales o sensibles aparecen, cuáles podrían faltar.  
6. **Limitaciones del modelo**: tipos de entidades que no detecta o que clasifica mal.  
7. **Relación con GDPR o con la norma local** (Colombia, México, Perú):  
   - ¿Qué principios de tratamiento se ven implicados (minimización, seguridad, finalidad, etc.)?

El propósito de este ejercicio es sirvir como puente entre la parte técnica (modelos NER) y la práctica profesional (arquitecturas de cumplimiento con LLM).

<a id="sec-recursos"></a>
## 12. Recursos y datasets recomendados

### 12.1. NER general en español

- Modelo `MMG/xlm-roberta-large-ner-spanish` (CoNLL-2002 ES):  
  - Página del modelo: https://huggingface.co/MMG/xlm-roberta-large-ner-spanish
- Introducción a token classification / NER en la Hugging Face course:  
  - https://huggingface.co/learn/llm-course/chapter7/2

### 12.2. Salud / medicina

- Repositorio de modelos biomédicos en español (PlanTL-GOB-ES):  
  - https://github.com/PlanTL-GOB-ES/lm-biomedical-clinical-es
- Datasets en Hugging Face:  
  - CANTEMIST NER: https://huggingface.co/datasets/PlanTL-GOB-ES/cantemist-ner
  - PharmaCoNER: https://huggingface.co/datasets/PlanTL-GOB-ES/pharmaconer

### 12.3. Legal

- Modelo NER legal en español:  
  - `agomez302/nlp-dr-ner`: https://huggingface.co/agomez302/nlp-dr-ner
- Guía de fine-tuning para clasificación de tokens en datos legales:  
  - https://huggingface.co/blog/bikashpatra/legal-data-token-classification-fine-tuning

### 12.4. Marco regulatorio

- GDPR – información general y guías:
  - https://gdpr.eu/what-is-gdpr/  
  - Ejemplos de contenidos para formación en GDPR: https://usercentrics.com/knowledge-hub/gdpr-training/
- Colombia – Ley 1581 de 2012:
  - Texto oficial: https://www.alcaldiabogota.gov.co/sisjur/normas/Norma1.jsp?i=49981  
- México – LFPDPPP: 
  - Guía de cumplimiento: https://resguard-solutions.com/blog/en/mexico-lfpdppp-data-protection-guide/  
- Perú – Ley 29733:
  - Guía de cumplimiento: https://resguard-solutions.com/blog/en/peru-law-29733-data-protection-guide/

<a id="sec-cierre"></a>
## 13. Cierre y reflexión final

En este notebook hemos visto cómo:

- Un modelo NER en español puede detectar entidades como personas, organizaciones y lugares en textos reales.
- Esa detección se puede usar para **anonimizar** textos antes de enviarlos a un LLM o a otra API, apoyando principios de **minimización** y **seguridad** recogidos en GDPR y en las leyes locales de Colombia, México y Perú.
- Los modelos generales tienen límites importantes, especialmente en dominios sensibles como salud y legal, donde suelen ser necesarios modelos NER **especializados** entrenados con datasets clínicos o jurídicos.
- Además de NER, usamos la tarea de **rellenar máscaras** (`fill-mask`) para observar cómo un modelo de lenguaje completa frases con huecos y cómo esto puede revelar **sesgos estadísticos** o hacer que ciertos datos enmascarados sean demasiado fáciles de adivinar.

La lección central es que la técnica por sí sola no garantiza cumplimiento:

- NER y `fill-mask` son **herramientas** dentro de un sistema más amplio que debe incluir políticas internas, formación, roles claros (como el DPO), contratos adecuados con proveedores y auditorías periódicas.
- Desde la perspectiva de diseño de aplicaciones de IA con LLM, el desafío está en combinar modelos, reglas de negocio y conocimiento legal para construir sistemas que sean útiles, pero también **seguros, no discriminatorios y alineados con la normativa vigente**.

Este es un Notebook en desarrollo y estos serían los temas a cubrir en la próxima sesión:

- Evaluar cuantitativamente la calidad de distintos modelos NER (precisión, recall, F1).  
- Integrar NER como paso previo en un pipeline RAG que filtre y anonimice documentos antes de indexarlos.  
- Usar sistemáticamente tareas tipo `fill-mask` para medir sesgos y riesgos de re-identificación en modelos de lenguaje.